In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import LabelEncoder



In [ ]:
dry_bean = fetch_openml(data_id=44225, as_frame=True)

X = dry_bean.data
y = dry_bean.target

# Create dataframe
bean_df = X.copy()
bean_df["Class"] = y

In [ ]:
print("Columns:\n", bean_df.columns)
print("\nFirst 5 rows:\n", bean_df.head())


In [ ]:
#combine into one datframe for easy analysis 
df = X.copy()
df["Class"] = y


In [ ]:
# Check columns
print(df.columns)

# Shape
print("Dataset Shape:", df.shape)

# Class distribution
print(df["Class"].value_counts())


In [ ]:
df

In [ ]:
# visualize relationship between area and parameter 
plt.figure(figsize=(10, 6))

classes = df["Class"].unique()

for bean_class in classes:
    subset = df[df["Class"] == bean_class]
    plt.scatter(
        subset["MEMBER_ANNUAL_INCOME"], 
        subset["ANNUAL_FEES"], 
        label=bean_class, 
        alpha=0.6
    )

plt.title("Income vs Annual Fees")
plt.xlabel("Member Annual Income")
plt.ylabel("Annual Fees")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Check missing values
print("Missing Values:\n")
print(df.isnull().sum())

In [ ]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

In [ ]:
for col in df.columns:
    
    # If column is numeric
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    
    # If column is categorical (string)
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# Check again
print("\nMissing Values After Cleaning:\n")
print(df.isnull().sum())



In [ ]:
# Convert date columns
if "START_DATE (YYYYMMDD)" in df.columns:
    df["START_DATE (YYYYMMDD)"] = pd.to_datetime(
        df["START_DATE (YYYYMMDD)"], errors="coerce"
    )

if "END_DATE  (YYYYMMDD)" in df.columns:
    df["END_DATE  (YYYYMMDD)"] = pd.to_datetime(
        df["END_DATE  (YYYYMMDD)"], errors="coerce"
    )


In [ ]:
# Final check
print("\nData Types:\n")
print(df.dtypes)

In [ ]:
df_model = df.copy()

# Convert datetime columns into numeric year/month/day features
df_model["START_YEAR"] = df_model["START_DATE (YYYYMMDD)"].dt.year
df_model["START_MONTH"] = df_model["START_DATE (YYYYMMDD)"].dt.month
df_model["START_DAY"] = df_model["START_DATE (YYYYMMDD)"].dt.day

df_model["END_YEAR"] = df_model["END_DATE  (YYYYMMDD)"].dt.year
df_model["END_MONTH"] = df_model["END_DATE  (YYYYMMDD)"].dt.month
df_model["END_DAY"] = df_model["END_DATE  (YYYYMMDD)"].dt.day

# Drop original date columns and ID-like columns
X = df_model.drop(columns=[
    "Class",
    "MEMBERSHIP_NUMBER",
    "AGENT_CODE",
    "START_DATE (YYYYMMDD)",
    "END_DATE  (YYYYMMDD)"
], errors="ignore")

y = df_model["Class"]

# Convert categorical columns into numeric
X = pd.get_dummies(X, drop_first=True)

# Encode target column
le = LabelEncoder()
y = le.fit_transform(y)


In [ ]:
#Model Building
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    bootstrap=True,
    oob_score=True,
    random_state=42
)

# Train model
rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)

# OOB Score
print("OOB Score:", rf_model.oob_score_)


In [ ]:

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)


In [ ]:

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45)

plt.title("Confusion Matrix")
plt.show()

In [ ]:

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:

feature_importances = pd.Series(rf_model.feature_importances_, index=X.columns)

top5 = feature_importances.sort_values(ascending=False).head(5)

print("\nTop 5 Important Features:\n")
print(top5)

In [ ]:
plt.figure(figsize=(8, 5))
top5.sort_values().plot(kind="barh")

plt.title("Top 5 Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.grid(axis="x")

plt.show()